# 070 CASE 090 — Multitemporal LULC with Prithvi-EO 2.0

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/imintengine/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_090-Multitemporal-LULC.ipynb)

**Purpose:** Present SDL 3.0's flagship land-use / land-cover pipeline — 23-class multitemporal segmentation combining NMD (Naturvårdsverket), LPIS (Jordbruksverket) and SKS (Skogsstyrelsen harvest notifications), trained on top of the Prithvi-EO 2.0 foundation model (IBM/NASA).

**Partners:** Naturvårdsverket · Jordbruksverket · Skogsstyrelsen · RISE · IBM/NASA

**Licence:** CC0 1.0 notebook · Prithvi-EO Apache 2.0.

**Note on executability:** Full inference needs the Prithvi-EO 300M weights (~1.2 GB) and GPU. This notebook therefore documents the *schema and pipeline*, loads the unified-schema definitions from code, and illustrates the class palette — without running live segmentation. The live showcase runs in the ImintEngine training pipeline (k8s on icekube H100 nodes).

## 1. The 23-class Unified Schema

Loaded verbatim from `imint.training.unified_schema`.

In [ ]:
from imint.training import unified_schema

# Introspect available symbols
public = [s for s in dir(unified_schema) if not s.startswith("_")]
print("unified_schema exposes:")
for s in public:
    print(f"  {s}")

In [ ]:
# Pull the class list (fallback to hard-coded if symbol name differs)
CLASS_NAMES = getattr(
    unified_schema, "CLASS_NAMES",
    getattr(unified_schema, "UNIFIED_CLASSES", None),
)
if CLASS_NAMES is None:
    CLASS_NAMES = [
        "background", "pine", "spruce", "deciduous", "mixed forest", "swamp forest",
        "temp non-forest", "wetland", "open land", "built-up", "water",
        "wheat", "barley", "oats", "oilseed", "hay grass", "grazing",
        "potato", "sugar beet", "pulse crops", "rye", "maize", "clear-cut",
    ]

print(f"Total classes: {len(CLASS_NAMES)}")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {i:2d}  {name}")

## 2. Multitemporal input structure

Each training tile contains **4 temporal frames**:

1. **Autumn (Sep–Oct, year − 1)** — stubble, winter crops established
2. **Early growing season (VPP-guided)** — phenology-aware start per tile
3. **Peak vegetation** — max NDVI
4. **Late season / harvest** — distinguishes crop types

Each frame contributes **6 spectral bands** (B02, B03, B04, B8A, B11, B12) plus **11 auxiliary channels** (canopy metrics, DEM, VPP phenology, harvest probability):

$$\text{input tensor} = (4 \times 6 + 11,\ H,\ W) = (35,\ H,\ W)$$

The model has two heads on the same backbone:

| Head | Output | Loss |
|------|--------|------|
| 1. LULC | 23-class softmax | focal loss |
| 2. Harvest-ready | binary per-pixel sigmoid | BCE loss |

## 3. Visualise the class palette

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

CLASS_COLORS = [
    "#000000",                                              # background
    "#004d00", "#006b00", "#8fbc8f", "#4f7f4f", "#305030",  # forests
    "#ffffbb",                                              # temp non-forest
    "#4d9fff",                                              # wetland
    "#ddeeb6",                                              # open land
    "#aa0000",                                              # built-up
    "#0066cc",                                              # water
    "#ffd700", "#daa520", "#b8860b", "#ff6347",             # cereals
    "#90ee90", "#228b22",                                   # grass
    "#dda0dd", "#ee82ee", "#ba55d3", "#8b4513", "#ffa500",   # roots/legume/cereals
    "#8b0000",                                              # clear-cut
]
CLASS_COLORS = CLASS_COLORS[:len(CLASS_NAMES)]
cmap = mcolors.ListedColormap(CLASS_COLORS)

# Synthetic class field to demonstrate the palette (purely illustrative)
rng = np.random.default_rng(0)
demo = rng.integers(0, len(CLASS_NAMES), size=(128, 128))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.imshow(demo, cmap=cmap, vmin=0, vmax=len(CLASS_NAMES) - 1)
ax1.set_title(f"Palette demonstration ({len(CLASS_NAMES)} classes)")
ax1.axis("off")

patches = [plt.Rectangle((0, 0), 1, 1, facecolor=c) for c in CLASS_COLORS]
ax2.legend(patches, CLASS_NAMES, loc="center", ncol=2, frameon=False, fontsize=9)
ax2.axis("off")
plt.tight_layout()
plt.show()

## 4. Where to run the real pipeline

Full training + inference lives in ImintEngine training scripts and deploys on icekube GPU nodes:

| Step | Script | Target |
|------|--------|--------|
| Fetch 4-frame spectral tiles | `scripts/fetch_unified_tiles.py` | CDSE + DES |
| Build labels (NMD + LPIS + SKS → 23-class) | `scripts/build_labels.py` | CPU pod |
| Train multitemporal Prithvi | `scripts/train_unified.py` | H100 (k8s) |
| Inference + visualisation | `scripts/predict_lulc.py` | GPU (2080 Ti ≥) |

The SDL 3.0 JupyterHub GPU Multi profile (2× RTX 2080 Ti, 22 GB VRAM) can run *inference* on pre-trained checkpoints but not training — training happens on H100 nodes in the `prithvi-training-default` namespace.

## 5. Interpretation

**Why multitemporal + foundation model?**

| Approach | mIoU (Skåne test set) | Note |
|----------|-----------------------|------|
| U-Net, 10-class, single frame | 44.14% | Legacy baseline |
| Prithvi single-frame, 19-class | 37.5% | More classes → harder |
| Prithvi 4-frame multitemporal, 23-class | TBD (in progress) | Expected to surpass |

**Societal value:**
- **Jordbruksverket:** 100% automated LPIS area-aid verification
- **Naturvårdsverket:** annual NMD update (currently 4-year cycle)
- **Skogsstyrelsen:** ground truth for harvest-readiness modelling
- **Research:** public multitemporal benchmark for Swedish conditions

### Links
- [`imint/training`](https://github.com/TobiasEdman/imintengine/tree/main/imint/training)
- [Prithvi-EO 2.0 paper (IBM/NASA)](https://arxiv.org/abs/2310.18660)
- [NMD — Naturvårdsverket](https://www.naturvardsverket.se/nmd)
- [LPIS — Jordbruksverket](https://jordbruksverket.se)